# Sprint 4 Experiment — Nemotron-3-Ultra-550B (NVIDIA API) × Failed Questions

Runs the 76 questions that Gemini 3.1 Flash-Lite answered **wrong or refused** against  
Nemotron-3-Ultra-550B via the NVIDIA integrate API (streaming + thinking enabled).

Set `PROMPT` in Cell 2, then **Run All**.

| Setting | Value |
|---|---|
| Model | nemotron-nvidia (nvidia/nemotron-3-ultra-550b-a55b) |
| Questions | gemini_simple_failures.csv (76 rows — tathybrid + finhybrid + music_structured) |
| Prompt | ← set in Cell 2 |
| Chunk size | 3000 (author default — fixed) |
| Chunk overlap | 300 (author default — fixed) |
| Top-K | 5 (author default — fixed) |
| Temperature | 0.0 (professor requirement — fixed) |
| Embedding | all-MiniLM-L6-v2 (local, free) |

## Cell 0 — Colab setup (run this first if on Google Colab, skip if running locally)

In [ ]:
import os, sys

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    # ── 1. Install dependencies ───────────────────────────────────────────
    %pip install -q openai chromadb langchain langchain-text-splitters \
                    sentence-transformers PyPDF2 together google-genai pandas

    # ── 2. Clone the repo (replace with your actual GitHub URL) ──────────
    GITHUB_REPO = 'https://github.com/YOUR_USERNAME/LLM_Benchmark_Team_Project_2026.git'
    if not os.path.exists('/content/LLM_Benchmark_Team_Project_2026'):
        !git clone {GITHUB_REPO} /content/LLM_Benchmark_Team_Project_2026

    PROJECT_ROOT = '/content/LLM_Benchmark_Team_Project_2026'

    # ── 3. Mount Google Drive if you stored PDFs there ────────────────────
    # from google.colab import drive
    # drive.mount('/content/drive')
    # Then update PDF_DIRS in Cell 2 to point to your Drive paths

    print('Colab setup complete.')
else:
    PROJECT_ROOT = '/Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026'
    print('Running locally — skipping Colab setup.')

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

## Cell 1 — Setup paths

In [ ]:
import sys, os

# PROJECT_ROOT is set by Cell 0 (Colab) or defaults to local path
if 'PROJECT_ROOT' not in dir():
    PROJECT_ROOT = '/Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026'

project_root = PROJECT_ROOT
sprint4_root = os.path.join(project_root, 'Sprint 4')
sprint3_uda  = os.path.join(project_root, 'Sprint 3', 'UDA-Benchmark')

for p in [sprint4_root, sprint3_uda]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(sprint3_uda)  # UDA code uses relative paths from this root
print(f'project_root : {project_root}')
print(f'sprint4_root : {sprint4_root}')
print(f'sprint3_uda  : {sprint3_uda}')
print(f'cwd          : {os.getcwd()}')

if not os.path.isdir(sprint3_uda):
    raise FileNotFoundError(f'sprint3_uda not found: {sprint3_uda}')

## Cell 2 — ★ CONFIGURE THIS ★

In [2]:
import pandas as pd

# ── Only change PROMPT ────────────────────────────────────────────────────────
MODEL_KEY = 'nemotron-nvidia'   # fixed — do not change
PROMPT    = 'simple'            # 'simple' (zero-shot) | 'cot' (chain-of-thought)
# ─────────────────────────────────────────────────────────────────────────────

FAILURES_CSV = os.path.join(
    sprint4_root,
    'experiments/gemini-3.1-flash-lite/results/gemini_simple_failures.csv'
)
OUTPUT_DIR = os.path.join(sprint4_root, f'experiments/{MODEL_KEY}/results')

UDA_PDF_BASE = os.path.join(sprint3_uda, 'dataset/src_doc_files_example')
PDF_DIRS = {
    'music_structured': os.path.join(UDA_PDF_BASE, 'music_docs'),
    'tathybrid':        os.path.join(UDA_PDF_BASE, 'tat_docs'),
    'finhybrid':        os.path.join(UDA_PDF_BASE, 'fin_docs'),
}

# Load failures list
df_failures = pd.read_csv(FAILURES_CSV, index_col='no')
print(f'Total failed questions to re-run: {len(df_failures)}')
print()
print(df_failures.groupby(['dataset', 'failure_type']).size().reset_index(name='count').to_string(index=False))
print()

# Verify all PDFs exist
print('PDF availability check:')
missing_pdfs = []
for _, row in df_failures.iterrows():
    pdf_dir  = PDF_DIRS.get(row['dataset'], '')
    doc_name = str(row['question_id']).split('_', 2)[-1] if row['dataset'] == 'music_structured' \
               else row.get('doc_name', '')
    pdf_path = os.path.join(pdf_dir, str(doc_name) + '.pdf')
    if not os.path.exists(pdf_path):
        missing_pdfs.append(f"{row['dataset']}/{doc_name}")

missing_pdfs = list(dict.fromkeys(missing_pdfs))
if missing_pdfs:
    print(f'  WARNING — {len(missing_pdfs)} PDFs not found:')
    for p in missing_pdfs:
        print(f'    {p}')
else:
    print('  All PDFs found.')

Total failed questions to re-run: 76

         dataset failure_type  count
       finhybrid      REFUSED     15
       finhybrid        WRONG     16
music_structured      REFUSED      3
music_structured        WRONG     13
       tathybrid    NO ANSWER      1
       tathybrid      REFUSED     20
       tathybrid        WRONG      8

PDF availability check:
  WARNING — 18 PDFs not found:
    tathybrid/
    finhybrid/
    music_structured/Q001
    music_structured/Q002
    music_structured/Q003
    music_structured/Q004
    music_structured/Q005
    music_structured/Q006
    music_structured/Q007
    music_structured/Q008
    music_structured/Q009
    music_structured/Q012
    music_structured/Q013
    music_structured/Q014
    music_structured/Q015
    music_structured/Q017
    music_structured/Q018
    music_structured/Q020


## Cell 3 — Verify failures CSV columns map correctly to RAGRunner inputs

In [3]:
# The failures CSV has: question_id, dataset, difficulty, failure_type,
#                       question, ground_truth, llm_answer
# RAGRunner.run() needs: question_id, question, ground_truth, doc_name
# For tathybrid/finhybrid: doc_name is embedded in question_id after the prefix
# For music_structured: single doc — music_dataset

import re

def extract_doc_name(row):
    ds  = row['dataset']
    qid = str(row['question_id'])
    if ds == 'music_structured':
        return 'music_dataset'
    elif ds == 'tathybrid':
        # S3_TATHYBRID_<q_uid> — doc_name is in notes or we look it up from combined CSV
        return row.get('doc_name', '')
    elif ds == 'finhybrid':
        # S3_FINHYBRID_ADI/2009/page_49.pdf-1 → doc_name = ADI/2009/page_49.pdf-1
        # but the PDF lives as ADI/2009/page_49.pdf → strip trailing -N suffix
        raw = re.sub(r'^S3_FINHYBRID_', '', qid)
        doc = re.sub(r'-\d+$', '', raw)          # remove trailing -1, -2, -3 etc.
        doc = re.sub(r'\.pdf$', '', doc)          # remove .pdf extension
        return doc
    return ''

# Load the full combined CSV to get doc_name for tathybrid rows
combined_csv = os.path.join(sprint4_root, 'benchmark/questions/all_questions_combined.csv')
df_combined  = pd.read_csv(combined_csv)
doc_name_map = df_combined.set_index('question_id')['doc_name'].to_dict()

df_failures = df_failures.copy()
df_failures['doc_name'] = df_failures.apply(
    lambda r: doc_name_map.get(r['question_id'], extract_doc_name(r)), axis=1
)

print('doc_name samples per dataset:')
for ds in df_failures['dataset'].unique():
    sub = df_failures[df_failures['dataset'] == ds]
    print(f'  {ds}:')
    for _, r in sub.head(3).iterrows():
        print(f'    {r["question_id"]}  →  doc_name={r["doc_name"]}')

print(f'\nMissing doc_names: {(df_failures["doc_name"] == "").sum()}')

doc_name samples per dataset:
  tathybrid:
    S3_TATHYBRID_cf649b7b26933abd29572697283fbf67  →  doc_name=inpixon_2019
    S3_TATHYBRID_f14766d60aaa9fc5ad134af01d563db4  →  doc_name=inpixon_2019
    S3_TATHYBRID_4a97a3a079b9d171bb3e3c05a5d558fa  →  doc_name=inpixon_2019
  finhybrid:
    S3_FINHYBRID_ABMD/2012/page_75.pdf-1  →  doc_name=ABMD_2012
    S3_FINHYBRID_ABMD/2012/page_75.pdf-2  →  doc_name=ABMD_2012
    S3_FINHYBRID_ABMD/2012/page_41.pdf-2  →  doc_name=ABMD_2012
  music_structured:
    S2_Q001  →  doc_name=music_dataset
    S2_Q002  →  doc_name=music_dataset
    S2_Q003  →  doc_name=music_dataset

Missing doc_names: 0


## Cell 4 — Run benchmark on failed questions (all 3 datasets)

In [4]:
from framework.rag_runner import RAGRunner
from datetime import datetime

os.makedirs(OUTPUT_DIR, exist_ok=True)
all_results = []

for dataset, group_df in df_failures.groupby('dataset'):
    print(f"\n{'='*60}")
    print(f'Dataset: {dataset}  ({len(group_df)} questions)')
    print(f"{'='*60}")

    pdf_dir = PDF_DIRS.get(dataset, '')
    if not pdf_dir or not os.path.isdir(pdf_dir):
        print(f'  SKIP — PDF directory not found: {pdf_dir}')
        continue

    # music_structured has no UDA eval — use nqtext as neutral runner dataset key
    runner_dataset = dataset if dataset != 'music_structured' else 'nqtext'

    runner = RAGRunner(model_key=MODEL_KEY, dataset=runner_dataset, prompt=PROMPT)

    # Write a temp CSV for this dataset
    tmp_csv = os.path.join(OUTPUT_DIR, f'_tmp_{dataset}.csv')
    group_df.reset_index().to_csv(tmp_csv, index=False)

    results_df = runner.run(
        questions_csv=tmp_csv,
        pdf_dir=pdf_dir,
        doc_col='doc_name',
        output_dir=OUTPUT_DIR,
    )
    results_df['dataset_actual'] = dataset
    all_results.append(results_df)
    os.remove(tmp_csv)

# Combine all datasets
results_combined = pd.concat(all_results, ignore_index=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
final_path = os.path.join(OUTPUT_DIR, f'ALL_RESULTS_nemotron_nvidia_{PROMPT}_{ts}.csv')
results_combined.to_csv(final_path, index=False)
print(f'\nAll results saved → {final_path}')
print(f'Total rows: {len(results_combined)}')

# Quick answer-rate summary
results_combined['is_empty'] = results_combined['response'].fillna('').str.strip() == ''
print()
print('Answer rate by dataset:')
for ds, grp in results_combined.groupby('dataset_actual'):
    n     = len(grp)
    empty = grp['is_empty'].sum()
    print(f'  {ds:<20}: {n - empty}/{n} answered  ({(n-empty)/n*100:.1f}%)')

/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/I772947/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/I772947/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and t


Dataset: finhybrid  (31 questions)


/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAGRunner ready: model=nemotron-nvidia, dataset=finhybrid, prompt=simple
  CHUNK_SIZE=3000, CHUNK_OVERLAP=300, TOP_K=5, TEMP=0.0

  PDF: ABMD_2012.pdf
  110 chunks indexed
  [1/7] during the 2012 year , did the equity awards in which the prescribed p...
    → The user is asking about whether during the 2012 year, the equity awards in whic
  [2/7] for equity awards where the performance criteria has been met in 2012 ...
    → The user is asking for the average compensation expense per year over which the 
  [3/7] did abiomed outperform the nasdaq medical equipment index?...
    → The answer is: Cannot be determined from the provided context. The report shows 
  [4/7] did abiomed outperform the nasdaq composite index?...
    → The answer is: Cannot be determined from the provided context. The context shows
  [5/7] how much of total future minimum lease payments are due currently?...
  [LLM ERROR] ResourceExhausted: Worker local total request limit reached (100/32)
    → 
  [6/7] what is 

## Cell 5 — Score tathybrid and finhybrid (UDA metrics)

In [5]:
import sys, os
sys.path.insert(0, sprint4_root)
sys.path.insert(0, sprint3_uda)

# Re-import after path setup
from Sprint_4_framework_score_gemini_simple import score_dataset, _load_tat_answers, _load_fin_answers

# Actually use the standalone scoring approach — avoids import path issues
import ast
import re

QA_BASE = os.path.join(sprint3_uda, 'dataset/qa/')

def _extract_raw_quid(row):
    notes = str(row.get('notes', '') or '')
    m = re.search(r'q_uid=(.+?)(?:;|$)', notes)
    if m:
        return m.group(1).strip()
    qid = str(row['question_id'])
    for prefix in ('S3_TATHYBRID_', 'S3_FINHYBRID_'):
        if qid.startswith(prefix):
            return qid[len(prefix):]
    return qid

def load_tat_answers(qa_path):
    df = pd.read_csv(qa_path, sep='|')
    out = {}
    for _, row in df.iterrows():
        raw = row['answer']
        try:
            answer_list = ast.literal_eval(str(raw))
        except Exception:
            answer_list = [str(raw)]
        scale = '' if pd.isna(row.get('answer_scale')) else str(row['answer_scale'])
        out[str(row['q_uid'])] = {'answer': answer_list, 'answer_type': str(row['answer_type']), 'scale': scale}
    return out

def load_fin_answers(qa_path):
    df = pd.read_csv(qa_path, sep='|')
    return {str(r['q_uid']): {'str_answer': str(r['answer_1']), 'exe_answer': str(r['answer_2'])}
            for _, r in df.iterrows()}

print('=== tathybrid ===')
tat_df   = results_combined[results_combined['dataset_actual'] == 'tathybrid'].copy()
tat_qa   = load_tat_answers(QA_BASE + 'tat_qa.csv')
tat_df['raw_quid'] = tat_df.apply(_extract_raw_quid, axis=1)

from uda.eval.utils.tat_eval import TaTQAEmAndF1
ev_tat = TaTQAEmAndF1()
for _, row in tat_df.iterrows():
    gt = tat_qa.get(row['raw_quid'], {'answer': [], 'answer_type': 'span', 'scale': ''})
    ev_tat({'response': str(row.get('response', '') or ''), 'answers': gt, 'q_uid': row['raw_quid']})
gem, gf1, _, _ = ev_tat.get_overall_metric()
print(f'  Numeracy F1 : {gf1*100:.2f}%')
print(f'  Exact Match : {gem*100:.2f}%')

print()
print('=== finhybrid ===')
fin_df   = results_combined[results_combined['dataset_actual'] == 'finhybrid'].copy()
fin_qa   = load_fin_answers(QA_BASE + 'fin_qa.csv')
fin_df['raw_quid'] = fin_df.apply(_extract_raw_quid, axis=1)

from uda.eval.utils.fin_eval import FinQAEm
ev_fin = FinQAEm()
for _, row in fin_df.iterrows():
    gt = fin_qa.get(row['raw_quid'], {'str_answer': '', 'exe_answer': ''})
    ev_fin({'response': str(row.get('response', '') or ''), 'answers': gt, 'q_uid': row['raw_quid']})
gem_fin = ev_fin.get_overall_metric()
print(f'  Exact Match : {gem_fin*100:.2f}%')

ModuleNotFoundError: No module named 'Sprint_4_framework_score_gemini_simple'

## Cell 6 — Compare Nemotron vs Gemini on the same failed questions

In [ ]:
# Load the original Gemini failures (ground truth for comparison)
gemini_failures = pd.read_csv(
    os.path.join(sprint4_root, 'experiments/gemini-3.1-flash-lite/results/gemini_simple_failures.csv'),
    index_col='no'
)

# Merge on question_id
comparison = gemini_failures[['question_id', 'dataset', 'failure_type', 'question', 'ground_truth']].copy()
comparison = comparison.rename(columns={'failure_type': 'gemini_failure'})

nemotron_answers = results_combined[['question_id', 'response']].rename(
    columns={'response': 'nemotron_answer'}
)
comparison = comparison.merge(nemotron_answers, on='question_id', how='left')

# Simple heuristic: did Nemotron refuse?
refusal_phrases = ['not contain', 'not provided', 'not available', 'not mentioned',
                   'not applicable', 'not specified', 'does not provide',
                   'cannot be determined', 'not in the', 'not include', 'not explicitly']
comparison['nemotron_refused'] = comparison['nemotron_answer'].fillna('').apply(
    lambda x: any(p in x.lower() for p in refusal_phrases)
)
comparison['nemotron_empty'] = comparison['nemotron_answer'].fillna('').str.strip() == ''

print('=== Nemotron response overview on 76 previously-failed questions ===')
print(f"  Answered (non-empty) : {(~comparison['nemotron_empty']).sum()}")
print(f"  Empty                : {comparison['nemotron_empty'].sum()}")
print(f"  Refused/Not found    : {comparison['nemotron_refused'].sum()}")
print()

# Save comparison CSV
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
comp_path = os.path.join(OUTPUT_DIR, f'gemini_vs_nemotron_comparison_{ts}.csv')
comparison.to_csv(comp_path, index=False)
print(f'Comparison saved → {comp_path}')
comparison.head(10)